# Chronos-2 - CATSA Train -> EmpaticaE4 Test (ALL signals @ 16 Hz)

**Signals (7 channels @ 16 Hz)**: ACC_x, ACC_y, ACC_z, EDA, TEMP, HR, HRV
- ACC: 32 Hz → 16 Hz  |  EDA, TEMP: 4 Hz → 16 Hz  |  HR/HRV: derived from BVP (64 Hz) → 16 Hz

**Model**: Amazon Chronos-2 (Chronos-T5-small, fine-tuning)
- Foundation time-series model (T5-based, d_model=512)
- 7-channel encoding: each channel encoded independently → embeddings concatenated
- Window 60 s = 960 samples @ 16 Hz; subsample=2 → 480 samples fed to Chronos (within 512 context limit)
- Head: Linear(512×7=3584, 256) → GELU → Dropout → Linear(256, 64) → GELU → Dropout → Linear(64, 1)

**Train**: CATSA | **Test**: EmpaticaE4Stress subject_01~06

---
Install requirement:
- `pip install chronos-forecasting`

In [ ]:
import warnings
from copy import deepcopy
from pathlib import Path
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.signal import find_peaks

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

warnings.filterwarnings('ignore', category=FutureWarning)

TASKS        = ['Baseline', 'Logic', 'Nback', 'Stroop', 'Sudoku']
STRESS_TASKS = {'Logic', 'Nback', 'Stroop', 'Sudoku'}

FS_TARGET  = 16   # unified output rate
FS_BVP     = 64   # BVP source
FS_ACC_SRC = 32   # ACC source
FS_SLOW    = 4    # EDA / TEMP source
N_CH       = 7    # channels: ACC_x, ACC_y, ACC_z, EDA, TEMP, HR, HRV

CATSA_ROOT  = Path('/home/binghin2/Myproject/Dataset/CATSA')
E4_ROOT     = Path('/home/binghin2/Myproject/Dataset/EmpaticaE4Stress/Subjects')
SUBJECTS_E4 = [f'subject_{i:02d}' for i in range(1, 7)]

SAVE_DIR = Path('/home/binghin2/Myproject/Research/CATSA/Train/Individual_data/ALL/Save_model_Chronos2')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def discover_subjects(root: Path) -> List[str]:
    out = []
    for d in sorted(root.glob('Sub*'), key=lambda p: int(p.name[3:])):
        if d.is_dir() and all((d / t / 'BVP.csv').exists() for t in TASKS):
            out.append(d.name)
    return out


def resample_to_length(sig: np.ndarray, target_len: int) -> np.ndarray:
    sig = np.asarray(sig, dtype=np.float32).reshape(-1)
    if target_len <= 0: return np.empty(0, np.float32)
    if len(sig) == 0:   return np.zeros(target_len, np.float32)
    if len(sig) == target_len: return sig
    return np.interp(np.linspace(0, 1, target_len), np.linspace(0, 1, len(sig)), sig).astype(np.float32)


def bvp_to_hr_hrv(bvp: np.ndarray, fs: int = 64, out_fs: int = 4) -> Tuple[np.ndarray, np.ndarray]:
    sig = np.asarray(bvp, dtype=np.float32).reshape(-1)
    peaks, _ = find_peaks(sig, distance=int(fs * 0.30))
    n_out = len(sig) // (fs // out_fs)
    t_out = np.arange(n_out) / float(out_fs)
    if len(peaks) < 3:
        return np.zeros(n_out, np.float32), np.zeros(n_out, np.float32)
    t_peaks = peaks / float(fs)
    ibi = np.clip(np.diff(t_peaks), 1e-3, None)
    hr_out  = np.interp(t_out, t_peaks[1:], 60.0 / ibi).astype(np.float32)
    hrv_std = pd.Series(ibi).rolling(10, min_periods=1).std().fillna(0).values
    hrv_out = np.interp(t_out, t_peaks[1:], hrv_std).astype(np.float32)
    return hr_out, hrv_out


def _to_16hz(acc: np.ndarray, bvp: np.ndarray,
             eda: np.ndarray, temp: np.ndarray) -> np.ndarray:
    hr4, hrv4 = bvp_to_hr_hrv(bvp, fs=FS_BVP, out_fs=FS_SLOW)
    dur = min(min(len(eda), len(temp), len(hr4), len(hrv4)) / FS_SLOW,
              len(acc) / FS_ACC_SRC)
    if dur <= 0: raise ValueError('Empty signal')
    T = int(dur * FS_TARGET); ns = int(dur * FS_SLOW); na = int(dur * FS_ACC_SRC)
    ch = [resample_to_length(acc[:na, i], T) for i in range(3)] + [
         resample_to_length(eda[:ns],  T),
         resample_to_length(temp[:ns], T),
         resample_to_length(hr4[:ns],  T),
         resample_to_length(hrv4[:ns], T)]
    return np.stack(ch, axis=1)  # [T, 7]


def load_catsa_16hz(task_dir: Path) -> np.ndarray:
    return _to_16hz(
        pd.read_csv(task_dir / 'ACC.csv').values[:, :3].astype(np.float32),
        pd.read_csv(task_dir / 'BVP.csv').values.reshape(-1).astype(np.float32),
        pd.read_csv(task_dir / 'EDA.csv').values.reshape(-1).astype(np.float32),
        pd.read_csv(task_dir / 'TEMP.csv').values.reshape(-1).astype(np.float32),
    )


def load_e4_16hz(subject_dir: Path) -> np.ndarray:
    kw = dict(header=None, skiprows=2)
    return _to_16hz(
        pd.read_csv(subject_dir / 'ACC.csv',  **kw).values[:, :3].astype(np.float32),
        pd.read_csv(subject_dir / 'BVP.csv',  **kw).values.reshape(-1).astype(np.float32),
        pd.read_csv(subject_dir / 'EDA.csv',  **kw).values.reshape(-1).astype(np.float32),
        pd.read_csv(subject_dir / 'TEMP.csv', **kw).values.reshape(-1).astype(np.float32),
    )


def build_catsa_arrays(subjects: List[str], W: int, S: int) -> Tuple[np.ndarray, np.ndarray]:
    xs, ys = [], []
    for sub in subjects:
        sub_dir = CATSA_ROOT / sub
        segs = []
        for task in TASKS:
            try: segs.append(load_catsa_16hz(sub_dir / task))
            except Exception: continue
        if not segs: continue
        cat = np.concatenate(segs, axis=0)
        mu = cat.mean(0); sd = cat.std(0) + 1e-8
        for task in TASKS:
            try:
                sig = ((load_catsa_16hz(sub_dir / task) - mu) / sd).astype(np.float32)
                lab = int(task in STRESS_TASKS)
                for i in range(0, len(sig) - W + 1, S):
                    xs.append(sig[i:i+W].T); ys.append(lab)
            except Exception: continue
    if not xs:
        return np.empty((0, N_CH, W), np.float32), np.empty(0, np.int64)
    return np.asarray(xs, np.float32), np.asarray(ys, np.int64)


def build_e4_labels(n: int, fs: int = FS_TARGET) -> Tuple[np.ndarray, int]:
    fixed = {'rest0':180,'task1':600,'rest1':120,'task2':300,'rest2':120,
             'task3':180,'rest3':120,'rest4':120,'task5':60,'rest5':120}
    task4_s = max(0, n // fs - sum(fixed.values()))
    segs = [('rest0',180,0),('task1',600,1),('rest1',120,-1),('task2',300,1),
            ('rest2',120,-1),('task3',180,1),('rest3',120,-1),('task4',task4_s,1),
            ('rest4',120,-1),('task5',60,1),('rest5',120,-1)]
    labels = np.full(n, -1, np.int64); cur = 0
    for _, dur_s, lab in segs:
        if dur_s <= 0: continue
        end = min(cur + dur_s * fs, n)
        if end > cur: labels[cur:end] = lab
        cur = end
        if cur >= n: break
    return labels, int(task4_s)


def e4_windows(signals: np.ndarray, labels: np.ndarray,
               W: int, S: int) -> Tuple[np.ndarray, np.ndarray]:
    n = min(len(signals), len(labels))
    signals, labels = signals[:n], labels[:n]
    mask = labels >= 0
    mu = signals[mask].mean(0) if mask.any() else signals.mean(0)
    sd = (signals[mask].std(0)+1e-8) if mask.any() else (signals.std(0)+1e-8)
    sig_n = ((signals - mu) / sd).astype(np.float32)
    xs, ys = [], []
    for i in range(0, len(sig_n) - W + 1, S):
        u = np.unique(labels[i:i+W])
        if len(u) == 1 and u[0] in (0, 1):
            xs.append(sig_n[i:i+W].T); ys.append(int(u[0]))
    if not xs:
        return np.empty((0, N_CH, W), np.float32), np.empty(0, np.int64)
    return np.asarray(xs, np.float32), np.asarray(ys, np.int64)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__(); self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, y):
        y = y.float()
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction='none')
        p = torch.sigmoid(logits)
        pt = p*y + (1-p)*(1-y)
        fl = (1-pt).pow(self.gamma) * bce
        if self.alpha is not None:
            fl = (self.alpha*y + (1-self.alpha)*(1-y)) * fl
        return fl.mean()


def make_loader(x, y, bs, shuffle):
    return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y).float()),
                      batch_size=bs, shuffle=shuffle)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval(); tot = n = tp = tn = fp = fn = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        lg = model(xb); tot += criterion(lg, yb).item()*len(xb); n += len(xb)
        p = (torch.sigmoid(lg) >= 0.5).long(); yi = yb.long()
        tp+=int(((p==1)&(yi==1)).sum()); tn+=int(((p==0)&(yi==0)).sum())
        fp+=int(((p==1)&(yi==0)).sum()); fn+=int(((p==0)&(yi==1)).sum())
    acc=(tp+tn)/max(n,1); pre=tp/max(tp+fp,1); rec=tp/max(tp+fn,1)
    return {'loss':tot/max(n,1),'accuracy':acc,'f1':2*pre*rec/max(pre+rec,1e-8),'precision':pre,'recall':rec}


def train_epoch_chronos(model, loader, optimizer, criterion):
    model.train(); tot = n = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        # bf16은 GradScaler 불필요 (fp16과 다름)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(xb)
            loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tot += loss.item()*len(xb); n += len(xb)
    return tot/max(n,1)


print('Utilities loaded.')

Device: cuda
Utilities loaded.


In [2]:
# Chronos-2 model (ALL signals, n_channels=7, subsample=2 → 480 samples per channel)
from chronos import ChronosPipeline

print('Loading pretrained Chronos-T5-small...')
pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map='cpu',
    torch_dtype=torch.float32,
)
print(f'Chronos context_length : {pipeline.model_context_length}')
print(f'T5 d_model             : {pipeline.model.model.config.d_model}')
print(f'T5 encoder layers      : {pipeline.model.model.config.num_layers}')


class Chronos2Classifier(nn.Module):
    def __init__(self, chronos_pipeline: ChronosPipeline,
                 n_channels: int = 7, subsample: int = 4, dropout: float = 0.3):
        super().__init__()
        self.subsample  = subsample
        self.n_channels = n_channels
        self.tokenizer  = chronos_pipeline.tokenizer
        self.chron_model = chronos_pipeline.model
        d_model = chronos_pipeline.model.model.config.d_model  # 512

        # Freeze all but the top 2 encoder blocks
        n_total = len(chronos_pipeline.model.model.encoder.block)
        n_freeze = max(0, n_total - 2)
        for i, blk in enumerate(chronos_pipeline.model.model.encoder.block):
            if i < n_freeze:
                for p in blk.parameters():
                    p.requires_grad = False
        print(f'Frozen {n_freeze}/{n_total} encoder blocks. Fine-tuning top {n_total - n_freeze}.')

        # Head: d_model * n_channels = 512*7 = 3584
        self.head = nn.Sequential(
            nn.LayerNorm(d_model * n_channels),
            nn.Linear(d_model * n_channels, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def _encode_channel(self, ch: torch.Tensor) -> torch.Tensor:
        # ch: [B, T_sub]
        device = ch.device
        token_ids, attn_mask, _ = self.tokenizer.context_input_transform(ch.detach().cpu())
        token_ids = token_ids.to(device)
        attn_mask = attn_mask.to(device)
        enc = self.chron_model.encode(input_ids=token_ids, attention_mask=attn_mask)
        mask_f = attn_mask.unsqueeze(-1).float()
        emb = (enc * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        return emb  # [B, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, 7, 960] -> subsample -> [B, 7, 480]
        x_sub = x[:, :, ::self.subsample]
        embeds = [self._encode_channel(x_sub[:, c, :]) for c in range(self.n_channels)]
        return self.head(torch.cat(embeds, dim=-1)).squeeze(-1)


model = Chronos2Classifier(pipeline, n_channels=N_CH, subsample=2, dropout=0.3).to(DEVICE)

x_dummy = torch.randn(2, N_CH, FS_TARGET*60, device=DEVICE)
out_test = model(x_dummy)
print(f'Input: {x_dummy.shape}  Output: {out_test.shape}')
print(f'Total params:     {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
del x_dummy, out_test

/home/binghin2/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pretrained Chronos-T5-small...


`torch_dtype` is deprecated! Use `dtype` instead!


Chronos context_length : 512
T5 d_model             : 512
T5 encoder layers      : 6
Frozen 4/6 encoder blocks. Fine-tuning top 2.
Input: torch.Size([2, 7, 960])  Output: torch.Size([2])
Total params:     47,095,681
Trainable params: 34,508,417


In [ ]:
# Hyperparameters & CATSA training
HP = {'seed':42, 'val_ratio':0.10, 'window_size':FS_TARGET*60, 'stride':FS_TARGET*10,
      'batch_size':16, 'epochs':25, 'lr':5e-5, 'wd':1e-4, 'patience':6,
      'subsample':2, 'dropout':0.3}

np.random.seed(HP['seed']); torch.manual_seed(HP['seed'])
if torch.cuda.is_available(): torch.cuda.manual_seed_all(HP['seed'])

subjects = discover_subjects(CATSA_ROOT)
print(f'CATSA subjects: {len(subjects)}')

rng = np.random.default_rng(HP['seed'])
idx = rng.permutation(len(subjects))
n_val = max(4, int(len(subjects)*HP['val_ratio']))
val_subs   = [subjects[i] for i in sorted(idx[:n_val])]
train_subs = [subjects[i] for i in sorted(idx[n_val:])]
print(f'Train: {len(train_subs)} | Val: {len(val_subs)}')

W, S = HP['window_size'], HP['stride']
print('Building arrays...')
x_tr, y_tr = build_catsa_arrays(train_subs, W, S)
x_va, y_va = build_catsa_arrays(val_subs,   W, S)
print(f'Train {x_tr.shape}  Val {x_va.shape}')

loader_tr = make_loader(x_tr, y_tr, HP['batch_size'], True)
loader_va = make_loader(x_va, y_va, HP['batch_size'], False)

alpha = float((len(y_tr) - y_tr.sum()) / len(y_tr))
criterion = FocalLoss(gamma=2.0, alpha=alpha).to(DEVICE)

# Separate LRs: encoder params get 10x slower lr than head
encoder_params = list(model.chron_model.parameters())
head_params    = list(model.head.parameters())
optimizer = torch.optim.Adam(
    [
        {'params': [p for p in encoder_params if p.requires_grad], 'lr': HP['lr'] * 0.1},
        {'params': head_params, 'lr': HP['lr']},
    ],
    weight_decay=HP['wd'],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3)

best_val, best_state, best_epoch, wait, history = float('inf'), None, 0, 0, []

for ep in range(1, HP['epochs']+1):
    tr = train_epoch_chronos(model, loader_tr, optimizer, criterion)
    va = evaluate(model, loader_va, criterion)
    scheduler.step(va['loss'])
    history.append({'epoch':ep,'train_loss':tr,'val_loss':va['loss'],'val_f1':va['f1']})
    if va['loss'] < best_val:
        best_val=va['loss']; best_epoch=ep; wait=0
        best_state = deepcopy(model.state_dict())
        torch.save({'model_state_dict':best_state,'hp':HP,'epoch':ep}, SAVE_DIR/'best_model.pt')
    else:
        wait += 1
        if wait >= HP['patience']:
            print(f'Early stop ep={ep} (best={best_epoch})'); break
    if ep%5==0 or ep==1:
        print(f'[{ep:3d}] train={tr:.4f}  val={va["loss"]:.4f}  f1={va["f1"]:.4f}')
model.load_state_dict(best_state)
print(f'Done. Best epoch={best_epoch}, val_loss={best_val:.4f}')

NameError: name 'FS_TARGET' is not defined

In [ ]:
# Evaluate on EmpaticaE4Stress
model.eval(); all_true, all_pred, rows = [], [], []

for sub in SUBJECTS_E4:
    try:
        sig = load_e4_16hz(E4_ROOT / sub)
    except Exception as e:
        print(f'{sub}: {e}'); continue
    labels, t4 = build_e4_labels(len(sig), fs=FS_TARGET)
    x, y_true  = e4_windows(sig, labels, W, S)
    if len(x) == 0:
        rows.append({'subject':sub,'task4_sec':t4,'n_windows':0,
                     'accuracy':float('nan'),'f1':float('nan'),
                     'precision':float('nan'),'recall':float('nan')}); continue
    with torch.no_grad():
        chunks = [(torch.sigmoid(model(torch.from_numpy(x[i:i+16]).to(DEVICE)))>=0.5)
                  .cpu().numpy().astype(int) for i in range(0,len(x),16)]
        y_pred = np.concatenate(chunks)
    rows.append({'subject':sub,'task4_sec':t4,'n_windows':len(y_true),
                 'accuracy':float(accuracy_score(y_true,y_pred)),
                 'f1':float(f1_score(y_true,y_pred,zero_division=0)),
                 'precision':float(precision_score(y_true,y_pred,zero_division=0)),
                 'recall':float(recall_score(y_true,y_pred,zero_division=0))})
    all_true.extend(y_true.tolist()); all_pred.extend(y_pred.tolist())

result_df = pd.DataFrame(rows)
print('=== Per-subject metrics ==='); display(result_df)

if all_true:
    ov_acc  = accuracy_score(all_true, all_pred)
    ov_f1   = f1_score(all_true, all_pred, zero_division=0)
    ov_prec = precision_score(all_true, all_pred, zero_division=0)
    ov_rec  = recall_score(all_true, all_pred, zero_division=0)
    print(f'\nOverall  Acc={ov_acc:.4f}  F1={ov_f1:.4f}  Prec={ov_prec:.4f}  Rec={ov_rec:.4f}')
    cm = confusion_matrix(all_true, all_pred)
    print('\nConfusion Matrix:\n', cm)
    print('\n', classification_report(all_true,all_pred,target_names=['Non-stress','Stress'],zero_division=0))

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot([h['epoch'] for h in history],[h['train_loss'] for h in history],label='Train',lw=2)
axes[0].plot([h['epoch'] for h in history],[h['val_loss']   for h in history],label='Val',  lw=2)
axes[0].set_title('Train/Val Loss (CATSA)'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(ls='--',alpha=0.4)

if all_true:
    bars = axes[1].bar(['Accuracy','F1','Precision','Recall'],
                       [ov_acc,ov_f1,ov_prec,ov_rec],
                       color=['#4C78A8','#F58518','#54A24B','#E45756'])
    axes[1].set_ylim(0,1); axes[1].set_title('Overall Metrics on EmpaticaE4')
    axes[1].grid(axis='y',ls='--',alpha=0.4)
    for b,v in zip(bars,[ov_acc,ov_f1,ov_prec,ov_rec]):
        axes[1].text(b.get_x()+b.get_width()/2,v+0.02,f'{v:.4f}',ha='center',va='bottom',fontsize=9)
    cm_n = cm.astype(float)/cm.sum(1,keepdims=True).clip(1)
    im = axes[2].imshow(cm_n,cmap='Blues',vmin=0,vmax=1)
    axes[2].set_title('Normalized Confusion Matrix')
    axes[2].set_xticks([0,1]); axes[2].set_yticks([0,1])
    axes[2].set_xticklabels(['Non-stress','Stress']); axes[2].set_yticklabels(['Non-stress','Stress'])
    axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')
    thr = cm_n.max()/2
    for i in range(2):
        for j in range(2):
            axes[2].text(j,i,f'{cm_n[i,j]:.3f}\n({cm[i,j]})',ha='center',va='center',
                         color='white' if cm_n[i,j]>thr else 'black',fontsize=11)
    fig.colorbar(im,ax=axes[2],fraction=0.046,pad=0.04)

fig.suptitle('Chronos-2 (Chronos-T5-small) - CATSA->EmpaticaE4 (ALL 7ch @ 16Hz)',fontsize=13,fontweight='bold')
plt.tight_layout()
plt.savefig(SAVE_DIR/'results_all_chronos2.png',dpi=150,bbox_inches='tight')
plt.show(); print('Saved results_all_chronos2.png')